# APM Chart - Analyse Exploratoire des Données (EDA)

## Description du Dataset
Ce notebook analyse le fichier **APM_Chart_ML.csv**.

**Caractéristiques principales:**
- **525,661 lignes** × **26 colonnes**
- Données temporelles à fréquence **1 minute**
- Variables: températures stator, puissance, tension, fréquence

## 1. Import des Librairies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Librairies importées avec succès!')

## 2. Chargement et Aperçu des Données

In [ ]:
df = pd.read_csv('../LAST_DATA/APM_Chart_ML.csv')

print('=' * 80)
print('APERÇU DES DONNÉES')
print('=' * 80)
df.head(10)

In [ ]:
print('\nINFORMATIONS SUR LE DATASET:')
df.info()

## 3. Dimensions et Types de Colonnes

In [ ]:
print('=' * 80)
print('DIMENSIONS')
print('=' * 80)
print(f'Nombre de lignes: {df.shape[0]:,}')
print(f'Nombre de colonnes: {df.shape[1]}')
print(f'\nMémoire: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

print('\n' + '=' * 80)
print('COLONNES')
print('=' * 80)
for i, col in enumerate(df.columns):
    print(f'{i+1:2}. {col} ({df[col].dtype})')

## 4. Résumé Statistique

In [ ]:
print('STATISTIQUES DESCRIPTIVES')
df.describe().T

## 5. Analyse des Valeurs Manquantes

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

print('VALEURS MANQUANTES:')
missing_df = pd.DataFrame({'Manquantes': missing, '%': missing_pct})
print(missing_df)
print(f'\nTotal: {missing.sum()} valeurs manquantes')

## 6. Distribution des Variables Principales

In [ ]:
# Variables principales - utiliser les colonnes qui existent
main_vars = ['MODE_TAG_1', 'REACTIVE_LOAD', 'SPEED_CTRL_pct', 'TERMINAL_VOLTAGE_kV', 'FREQUENCY_Hz']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, var in enumerate(main_vars):
    ax = axes[i]
    sns.histplot(df[var], kde=True, ax=ax, color='steelblue', alpha=0.7)
    ax.set_title(f'Distribution: {var}', fontsize=11, fontweight='bold')
    ax.axvline(df[var].mean(), color='red', linestyle='--', label=f'Moy: {df[var].mean():.2f}')
    ax.legend(fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('Distribution des Variables Principales', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Températures du stator (9 capteurs originaux)
temp_cols = [col for col in df.columns if col.startswith('STATOR_PHASE') and 'WINDING_TEMP' in col]
temp_cols = temp_cols[:9]  # Limiter à 9 pour 3x3 grid

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(temp_cols):
    ax = axes[i]
    sns.histplot(df[col], kde=True, ax=ax, color='coral', alpha=0.7)
    ax.set_title(col.replace('STATOR_', '').replace('_degC', ''), fontsize=9)

plt.suptitle('Températures du Stator', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Matrice de Corrélation

In [ ]:
numeric_cols = [col for col in df.select_dtypes(include=[np.number]).columns 
                if col not in ['Year', 'Month', 'Day', 'Hour', 'Minute', 'DayOfWeek', 'Quarter']]

corr = df[numeric_cols].corr()

plt.figure(figsize=(16, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, annot_kws={'size': 8})
plt.title('Matrice de Corrélation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Visualisation Temporelle

In [ ]:
if all(col in df.columns for col in ['Year', 'Month', 'Day', 'Hour', 'Minute']):
    df['DateTime'] = pd.to_datetime(df[['Year', 'Month', 'Day', 'Hour', 'Minute']])
    
    # Sous-échantillonner pour visualisation
    step = max(1, len(df) // 5000)
    sample = df.iloc[::step].copy()
    
    fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
    
    axes[0].plot(sample['DateTime'], sample['MODE_TAG_1'], color='blue', linewidth=0.5)
    axes[0].set_ylabel('Puissance (MW)')
    axes[0].set_title('Puissance Active')
    axes[0].fill_between(sample['DateTime'], sample['MODE_TAG_1'], alpha=0.3)
    
    temp_mean = sample[temp_cols].mean(axis=1)
    axes[1].plot(sample['DateTime'], temp_mean, color='red', linewidth=0.5)
    axes[1].set_ylabel('Temp (°C)')
    axes[1].set_title('Température Moyenne Stator')
    
    axes[2].plot(sample['DateTime'], sample['FREQUENCY_Hz'], color='green', linewidth=0.5)
    axes[2].axhline(50, color='red', linestyle='--', label='50 Hz')
    axes[2].set_ylabel('Fréquence (Hz)')
    axes[2].set_xlabel('Date')
    axes[2].legend()
    
    plt.suptitle('Évolution Temporelle', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# Analyse par heure
if 'Hour' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    hourly_power = df.groupby('Hour')['MODE_TAG_1'].mean()
    axes[0].bar(hourly_power.index, hourly_power.values, color='steelblue', edgecolor='black')
    axes[0].set_xlabel('Heure')
    axes[0].set_ylabel('Puissance Moyenne (MW)')
    axes[0].set_title('Puissance par Heure')
    
    hourly_temp = df.groupby('Hour')[temp_cols].mean().mean(axis=1)
    axes[1].bar(hourly_temp.index, hourly_temp.values, color='coral', edgecolor='black')
    axes[1].set_xlabel('Heure')
    axes[1].set_ylabel('Température Moyenne (°C)')
    axes[1].set_title('Température par Heure')
    
    plt.tight_layout()
    plt.show()

## 9. Détection des Outliers (Box Plots)

In [ ]:
# Box plots - échantillonner pour performance
sample_box = df.sample(n=min(50000, len(df)), random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

boxplot_vars = ['MODE_TAG_1', 'REACTIVE_LOAD', 'TERMINAL_VOLTAGE_kV', 
                'FREQUENCY_Hz', 'AMBIENT_AIR_TEMP_C', 'SPEED_CTRL_pct']

for i, var in enumerate(boxplot_vars):
    axes[i].boxplot(sample_box[var].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='lightblue'))
    axes[i].set_title(var, fontsize=11, fontweight='bold')
    axes[i].set_ylabel(var)

plt.suptitle('Box Plots - Détection des Outliers', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots des températures
temp_data = df[temp_cols].melt(var_name='Capteur', value_name='Température')

plt.figure(figsize=(16, 6))
sns.boxplot(data=temp_data, x='Capteur', y='Température', palette='Reds')
plt.xticks(rotation=45, ha='right')
plt.title('Températures par Capteur', fontsize=14, fontweight='bold')
plt.ylabel('Température (°C)')
plt.tight_layout()
plt.show()

## 10. Relations entre Variables

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sample = df.sample(n=min(5000, len(df)), random_state=42)

# Puissance vs Température
axes[0].scatter(sample['MODE_TAG_1'], sample['STATOR_PHASE_A_WINDING_TEMP_1_degC'], 
                alpha=0.3, s=5, c='blue')
axes[0].set_xlabel('Puissance (MW)')
axes[0].set_ylabel('Température Phase A (°C)')
axes[0].set_title('Puissance vs Température')

# Tension vs Fréquence
axes[1].scatter(sample['TERMINAL_VOLTAGE_kV'], sample['FREQUENCY_Hz'], 
                alpha=0.3, s=5, c='green')
axes[1].set_xlabel('Tension (kV)')
axes[1].set_ylabel('Fréquence (Hz)')
axes[1].set_title('Tension vs Fréquence')

# Température ambiante vs Stator
axes[2].scatter(sample['AMBIENT_AIR_TEMP_C'], sample['STATOR_PHASE_A_WINDING_TEMP_1_degC'], 
                alpha=0.3, s=5, c='red')
axes[2].set_xlabel('Température Ambiante (°C)')
axes[2].set_ylabel('Température Stator (°C)')
axes[2].set_title('Ambiante vs Stator')

plt.suptitle('Relations entre Variables', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Pair plot
key_vars = ['MODE_TAG_1', 'STATOR_PHASE_A_WINDING_TEMP_1_degC', 'TERMINAL_VOLTAGE_kV', 'FREQUENCY_Hz']
sample_small = df[key_vars].sample(n=min(1000, len(df)), random_state=42)

g = sns.pairplot(sample_small, diag_kind='kde', plot_kws={'alpha': 0.5, 's': 15})
g.figure.suptitle('Pair Plot - Variables Principales', y=1.02, fontsize=14, fontweight='bold')
plt.show()

## 11. Conclusions

### Résumé:
- **Qualité des données**: Pas de valeurs manquantes
- **Puissance active**: 0-124 MW
- **Températures stator**: ~60-115°C
- **Fréquence**: ~50 Hz (nominal)
- **Corrélations fortes** entre températures des phases et avec la puissance